In [18]:
import requests
import zipfile
from google.colab import userdata
import pandas as pd

my_user_agent = userdata.get('SEC_USER_AGENT')

url = "https://www.sec.gov/files/dera/data/financial-statement-data-sets/2026q3.zip"
headers = {"User-Agent": my_user_agent}

response = requests.get(url, headers=headers, stream=True)

if response.status_code == 200:
    with open("2026q3.zip", "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print("Download complete")
else:
    print(f"Failed to download. Status code: {response.status_code}")

Download complete


In [19]:
with zipfile.ZipFile("2026q3.zip", 'r') as zip_ref:
    zip_ref.extractall()
print("Extraction complete! You should now see SUB.txt and NUM.txt in the file browser.")

Extraction complete! You should now see SUB.txt and NUM.txt in the file browser.


In [34]:
sub_df = pd.read_csv('sub.txt', sep='\t', low_memory=False)

# for 10-K / 10-Q filings
cols_to_keep_sub = ['adsh', 'name', 'cik', 'sic', 'form']
existing_cols_sub = [c for c in cols_to_keep_sub if c in sub_df.columns]
sub_df = sub_df[existing_cols_sub]

sub_df = sub_df[sub_df['form'].isin(['10-K', '10-Q'])]
print(f"Loaded {len(sub_df):,} target filings from sub.txt.")

Loaded 5,607 target filings from sub.txt.


In [35]:
num_df = pd.read_csv('num.txt', sep='\t', low_memory=False)

cols_to_keep_num = ['adsh', 'tag', 'version', 'ddate', 'qtrs', 'uom', 'value']
existing_cols_num = [c for c in cols_to_keep_num if c in num_df.columns]
num_df = num_df[existing_cols_num]

print(f"Loaded {len(num_df):,} raw facts from num.txt.")

Loaded 3,905,397 raw facts from num.txt.


In [36]:
merged_df = pd.merge(num_df, sub_df, on='adsh', how='inner')
print(f"Total matched facts: {len(merged_df):,}")


del num_df
del sub_df
gc.collect()

Total matched facts: 3,389,165


0

In [37]:
merged_df['ddate'] = pd.to_datetime(merged_df['ddate'], format='%Y%m%d', errors='coerce')

output_path = '2026_q3_financials_clean.parquet'
print(f"Exporting to {output_path}...")


merged_df.to_parquet(output_path, engine='pyarrow', compression='snappy')
print("File ready.")

display(merged_df.head())

Exporting to 2026_q3_financials_clean.parquet...
File ready.


,adsh,tag,version,ddate,qtrs,uom,value,name,cik,sic,form
0,0000002488-26-000123,AccountsPayableCurrent,us-gaap/2026,2025-12-31,0,USD,2.929000e+09,ADVANCED MICRO DEVICES INC,2488,3674.0,10-Q
1,0000002488-26-000123,AccountsPayableCurrent,us-gaap/2026,2026-06-30,0,USD,5.359000e+09,ADVANCED MICRO DEVICES INC,2488,3674.0,10-Q
2,0000002488-26-000123,AccountsReceivableNetCurrent,us-gaap/2026,2025-12-31,0,USD,6.315000e+09,ADVANCED MICRO DEVICES INC,2488,3674.0,10-Q
3,0000002488-26-000123,AccountsReceivableNetCurrent,us-gaap/2026,2026-06-30,0,USD,7.281000e+09,ADVANCED MICRO DEVICES INC,2488,3674.0,10-Q
4,0000002488-26-000123,AccruedLiabilitiesCurrentAndNoncurrent,us-gaap/2026,2025-12-31,0,USD,5.250000e+09,ADVANCED MICRO DEVICES INC,2488,3674.0,10-Q
